In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from groq import Groq

In [ ]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")
    
MODEL = "openai/gpt-oss-120b"
groq = Groq(api_key=groq_api_key)

DB = "products.db"

In [ ]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS products (name TEXT, platform TEXT, price REAL)")
    conn.commit()


In [ ]:
def set_product_info(name, platform, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO products (name, platform, price) VALUES (?, ?, ?)", (name.lower(), platform, price))
        conn.commit()

In [ ]:
product_info = [
    {"product": "Macbook", "platform": "Amazon", "price": 100000},
    {"product": "Macbook", "platform": "Flipkart", "price": 95000},
    {"product": "Macbook", "platform": "Croma", "price": 98000},
    {"product": "Macbook", "platform": "Reliance Digital", "price": 97500},

    {"product": "Iphone17", "platform": "Amazon", "price": 89000},
    {"product": "Iphone17", "platform": "Flipkart", "price": 90000},
    {"product": "Iphone17", "platform": "Croma", "price": 91500},
    {"product": "Iphone17", "platform": "Reliance Digital", "price": 90500},

    {"product": "Samsung Galaxy S25", "platform": "Amazon", "price": 74999},
    {"product": "Samsung Galaxy S25", "platform": "Flipkart", "price": 72999},
    {"product": "Samsung Galaxy S25", "platform": "Croma", "price": 73999},

    {"product": "Dell XPS 15", "platform": "Amazon", "price": 145000},
    {"product": "Dell XPS 15", "platform": "Flipkart", "price": 142000},
    {"product": "Dell XPS 15", "platform": "Croma", "price": 147000},

    {"product": "Sony WH-1000XM6", "platform": "Amazon", "price": 34999},
    {"product": "Sony WH-1000XM6", "platform": "Flipkart", "price": 33999},
    {"product": "Sony WH-1000XM6", "platform": "Croma", "price": 35500},

    {"product": "iPad Air", "platform": "Amazon", "price": 59900},
    {"product": "iPad Air", "platform": "Flipkart", "price": 57900},
    {"product": "iPad Air", "platform": "Croma", "price": 58900},

    {"product": "OnePlus 13", "platform": "Amazon", "price": 69999},
    {"product": "OnePlus 13", "platform": "Flipkart", "price": 67999},
    {"product": "OnePlus 13", "platform": "Reliance Digital", "price": 68999},
]

for product in product_info:
    set_product_info(product["product"], product["platform"], product["price"])


In [ ]:
system_prompt = """You are a helpful e-commerce AI assistant called Commy. 
You are able to answer questions about different products, their availability on different platforms and their prices. 
If you don't know the answer, just say so. Don't make up an answer."""

In [ ]:
def get_product_info(product_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM products WHERE name = ?", (product_name.lower(),))
        product = cursor.fetchall()
        return f"Details of {product_name}: Product Name: {product[0:][0]}, Platform: {product[0:][1]}, Price: {product[0:][2]}" if product else f"No details found for {product_name}"


In [ ]:
print(get_product_info("Washing Machine"))

In [ ]:
product_function = {
"name" : "get_product_info",
"description" : "Get information of platform and price of a product",
"parameters" : {
    "type" : "object",
    "properties" : {
        "product_name" : {"type" : "string", "description" : "The name of the product",
        },
    },
    "required" : ["product_name"], 
    "additionalProperties" : False
    }
}

In [ ]:
tools = [{"type":"function", "function":product_function}]

In [ ]:
def chat(message,history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_product_info":
            arguments = json.loads(tool_call.function.arguments)
            product_name = arguments.get('product_name')
            product_info = get_product_info(product_name)
            responses.append({
                "role": "tool",
                "content": product_info,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch(share=True)